In [0]:
# retail_sales_pipeline.py
import dlt
from pyspark.sql.functions import *

from pyspark.sql.types import *
sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("region", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", DoubleType(), True),
    StructField("sale_date", TimestampType(), True)
])
products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("discount_rate", DoubleType(), True)
])
# -------------------- CONFIG --------------------
RAW_SALES_PATH    = "abfss://raw@vaishnavidata.dfs.core.windows.net/Sales/"
RAW_PRODUCTS_PATH = "abfss://raw@vaishnavidata.dfs.core.windows.net/Products/"

# Schema/history locations (must be writable by the pipeline cluster)
SCHEMA_LOCATION_SALES    = "/dbfs/dlt_schema/retail/sales"    # DBFS path example
SCHEMA_LOCATION_PRODUCTS = "/dbfs/dlt_schema/retail/products" # DBFS path example

# Target Unity Catalog (catalog.schema) for tables
TARGET_SCHEMA = "retail_catalog_02.sales_analytics"

# Helper to produce full UC table names
def tbl(name):
    return f"{TARGET_SCHEMA}.{name}"

# -------------------- BRONZE (raw ingestion) --------------------
@dlt.table(
    name=tbl("bronze_sales"),
    comment="Bronze: raw sales ingested with Auto Loader (no transforms)",
    table_properties={"quality": "bronze"}
)
def bronze_sales():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            # .schema(sales_schema)
            # .option("inferSchema", "true")                         # avoid relying on inference here
            .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_SALES)
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("cloudFiles.schemaHints", "sale_id INT, region STRING, sales_amount DOUBLE")  # allow new columns to appear
            .load(RAW_SALES_PATH)
    )

@dlt.table(
    name=tbl("bronze_products"),
    comment="Bronze: raw product metadata ingested with Auto Loader (no transforms)",
    table_properties={"quality": "bronze"}
)
def bronze_products():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")   # change to 'csv' if your product files are CSV
            .option("header", "true")
            .schema(products_schema)   
            .option("cloudFiles.schemaLocation", SCHEMA_LOCATION_PRODUCTS)
            # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .load(RAW_PRODUCTS_PATH)
    )

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
DLTImportException                        Traceback (most recent call last)
File <command-8867542721480184>, line 2
      1 # retail_sales_pipeline.py
----> 2 import dlt
      3 import pyspark.sql.functions as F
      4 from pyspark.sql.types import *

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/hook.py:71, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     65 if not self._should_hint and (
     66     (module := sys.modules.get(absolute_name)) is not None and
     67     (fname := get_allowed_file_name_or_none(module)) is not None and
     68     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     69         absolute_name, float("inf")) and not self._should_hint):
     70     self._should_hint = True
---> 71 module = self._original_builtins_import(name, *args, **kwargs)
     72 if (fname := fname or get_allowed_file_name_or_none(m

In [0]:
@dlt.table(
    name="silver_sales",
    comment="Cleaned sales data (Silver)"
)
@dlt.expect_or_drop("non_null_orderid", "sale_id IS NOT NULL")
@dlt.expect("valid_quantity", "quantity >= 0")
def silver_sales():
    return (
        dlt.read("bronze_sales")
        .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))
        .filter(col("sale_id").isNotNull())
    )

@dlt.table(
    name="silver_products",
    comment="Cleaned products data (Silver)"
)
@dlt.expect_or_drop("non_null_productid", "product_id IS NOT NULL")
def silver_products():
    return (
        dlt.read("bronze_products")
        .filter(col("product_id").isNotNull())
    )

# ======================================================
# 🔹 GOLD (Business Aggregates)
# ======================================================
@dlt.table(
    name="gold_daily_revenue",
    comment="Daily revenue per region (Gold)"
)
def gold_daily_revenue():
    return (
        dlt.read("silver_sales")
        .groupBy("region", "sale_date")
        .agg(
            sum(col("sales_amount")).alias("daily_revenue")  # ✅ clean alias
        )
    )

@dlt.table(
    name="gold_product_performance",
    comment="Aggregated product performance (Gold)"
)
def gold_product_performance():
    return (
        dlt.read("silver_sales")
        .join(dlt.read("silver_products"), on="product_id", how="left")
        .groupBy("product_id", "product_name", "category")
        .agg(
            sum("quantity").alias("total_units_sold"),
            count("sale_id").alias("total_orders")
        )
    )